In [1]:
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "openpyxl", "--user", "--quiet"])
subprocess.run([sys.executable, "-m", "pip", "install", "transformers==4.40.0", "--quiet"])

CompletedProcess(args=['/blue/prismap-ai-core/vnolan/conda/envs/APARI2_Models/bin/python', '-m', 'pip', 'install', 'transformers==4.40.0', '--quiet'], returncode=0)

In [2]:
import importlib
import site
importlib.reload(site)

import pandas as pd
import numpy as np
import torch
import re
import shutil
import openpyxl
from torch import nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from copy import copy
from openpyxl.styles import Font, Border, PatternFill, Alignment

# -------------------------------------------------------
# CONFIGURATION — update paths here when files change
# -------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

MODEL_NAME        = "emilyalsentzer/Bio_ClinicalBERT"
ORDINAL_MODEL_PATH = "/blue/prismap-ai-core/deehu/projects/QUAL_Score_Project/best_model_ordinal.pt"
BINARY_MODEL_PATH  = "/blue/prismap-ai-core/deehu/projects/QUAL_Score_Project/best_model_binary.pt"
FEEDBACK_COL       = "Feedback"
MAX_LEN            = 256   # must match max_len used during training

EXCEL_FILES = [
    {
        "path": "/blue/prismap-ai-core/deehu/projects/QUAL_Score_Project/EPA_Narratives_Faculty.xlsx",
        "sheet": 0
    },
    {
        "path": "/blue/prismap-ai-core/deehu/projects/QUAL_Score_Project/CASE_data.xlsx",
        "sheet": "EPA"
    },
    {
        "path": "/blue/prismap-ai-core/deehu/projects/QUAL_Score_Project/Feedback OU.xlsx",
        "sheet": 0
    },
]

/blue/prismap-ai-core/vnolan/conda/envs/APARI2_Models/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Using device: cuda


In [5]:
# -------------------------------------------------------
# ORDINAL MODEL ARCHITECTURE
# Evidence: ordinal 0-3 via OrdinalEvidenceHead
# Suggestion + Connection: binary 0/1
# Used for: five_point_qual_score (0-5)
# -------------------------------------------------------
class OrdinalEvidenceHead(nn.Module):
    def __init__(self, hidden_size, num_classes=4):
        super().__init__()
        self.linear = nn.Linear(hidden_size, num_classes - 1)

    def forward(self, x):
        logits = self.linear(x)
        prob = torch.sigmoid(logits)
        return prob


def ordinal_class_from_probs(prob):
    return torch.sum(prob >= 0.5, dim=1)


class ClinicalBERTMultiTaskOrdinal(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.evidence_head   = OrdinalEvidenceHead(hidden_size, num_classes=4)
        self.suggestion_head = nn.Linear(hidden_size, 1)
        self.connection_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        e_prob  = self.evidence_head(cls)
        s_logit = self.suggestion_head(cls).squeeze(-1)
        c_logit = self.connection_head(cls).squeeze(-1)
        return e_prob, s_logit, c_logit


# -------------------------------------------------------
# BINARY MODEL ARCHITECTURE
# Evidence: binary 0/1 via linear head
# Suggestion + Connection: binary 0/1
# Used for: three_point_qual_score (mQuAL, 0-3)
# -------------------------------------------------------
class ClinicalBERTMultiTaskBinary(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.evidence_head   = nn.Linear(hidden_size, 1)
        self.suggestion_head = nn.Linear(hidden_size, 1)
        self.connection_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        e_logit = self.evidence_head(cls).squeeze(-1)
        s_logit = self.suggestion_head(cls).squeeze(-1)
        c_logit = self.connection_head(cls).squeeze(-1)
        return e_logit, s_logit, c_logit

In [6]:
print("Loading tokenizer and models...")

# One shared tokenizer for both models
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded.")

# Load Ordinal model (five_point_qual_score)
model_ordinal = ClinicalBERTMultiTaskOrdinal().to(device)
model_ordinal.load_state_dict(
    torch.load(ORDINAL_MODEL_PATH, map_location=device, weights_only=False)
)
model_ordinal.eval()
print("Ordinal model loaded.")

# Load Binary model (three_point_qual_score)
model_binary = ClinicalBERTMultiTaskBinary().to(device)
model_binary.load_state_dict(
    torch.load(BINARY_MODEL_PATH, map_location=device, weights_only=False)
)
model_binary.eval()
print("Binary model loaded.")

Loading tokenizer and models...
Tokenizer loaded.
Ordinal model loaded.
Binary model loaded.


In [7]:
# -------------------------------------------------------
# INFERENCE DATASET
# -------------------------------------------------------
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(
            text, truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0)
        }

# -------------------------------------------------------
# JUNK DETECTION
# -------------------------------------------------------
def is_junk(text):
    if pd.isna(text):
        return True
    text = str(text).strip()
    if len(text) == 0:
        return True
    if re.fullmatch(r"[^a-zA-Z0-9]+", text):  # symbols only
        return True
    if re.fullmatch(r"[a-zA-Z0-9]", text):    # single character
        return True
    return False

In [8]:
# -------------------------------------------------------
# ORDINAL INFERENCE
# Evidence (0-3) + Suggestion (0/1) + Connection (0/1)
# → five_point_qual_score (0-5)
# -------------------------------------------------------
def run_ordinal_inference(texts):
    dataset = InferenceDataset(texts, tokenizer)
    loader = DataLoader(dataset, batch_size=16)
    all_five_point = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            e_prob, s_logit, c_logit = model_ordinal(
                input_ids, attention_mask
            )
            e_pred = ordinal_class_from_probs(e_prob).cpu().numpy()
            s_pred = (torch.sigmoid(s_logit) >= 0.5).cpu().numpy().astype(int)
            c_pred = (torch.sigmoid(c_logit) >= 0.5).cpu().numpy().astype(int)
            total_pred = e_pred + s_pred + c_pred
            all_five_point.extend(total_pred)
    return np.array(all_five_point)

In [9]:
# -------------------------------------------------------
# BINARY INFERENCE
# Binary Evidence (0/1) + Suggestion (0/1) + Connection (0/1)
# → three_point_qual_score (mQuAL, 0-3)
# -------------------------------------------------------
def run_binary_inference(texts):
    dataset = InferenceDataset(texts, tokenizer)
    loader = DataLoader(dataset, batch_size=16)
    all_three_point = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            e_logit, s_logit, c_logit = model_binary(
                input_ids, attention_mask
            )
            e_pred = (torch.sigmoid(e_logit) >= 0.5).cpu().numpy().astype(int)
            s_pred = (torch.sigmoid(s_logit) >= 0.5).cpu().numpy().astype(int)
            c_pred = (torch.sigmoid(c_logit) >= 0.5).cpu().numpy().astype(int)
            total_pred = e_pred + s_pred + c_pred
            all_three_point.extend(total_pred)
    return np.array(all_three_point)

In [10]:
# -------------------------------------------------------
# PROCESS EACH EXCEL FILE
# -------------------------------------------------------
for file_config in EXCEL_FILES:
    filepath = file_config["path"]
    sheet_name = file_config["sheet"]

    print(f"\nProcessing: {filepath}")
    df = pd.read_excel(filepath, sheet_name=sheet_name)
    print("Columns:", df.columns.tolist())
    print("Total rows:", len(df))

    # Get all feedback texts
    texts = df[FEEDBACK_COL].fillna("").tolist()

    # Run both models
    five_point_scores = run_ordinal_inference(texts)
    three_point_scores = run_binary_inference(texts)

    # Apply junk mask → None for blank/junk entries
    mask = df[FEEDBACK_COL].apply(is_junk)

    three_point_final = [
        None if mask.iloc[i] else int(three_point_scores[i])
        for i in range(len(three_point_scores))
    ]
    five_point_final = [
        None if mask.iloc[i] else int(five_point_scores[i])
        for i in range(len(five_point_scores))
    ]

    print(f"Valid (non-junk) rows scored: {sum(1 for x in three_point_final if x is not None)}")
    print(f"Blank/junk rows skipped: {mask.sum()}")

    # ── Copy original file to preserve all sheets + formatting ──
    output_path = filepath.replace(".xlsx", "_QUAL_scored.xlsx")
    shutil.copy(filepath, output_path)

    # ── Open copied file and write only to target sheet ──
    wb = openpyxl.load_workbook(output_path)

    if isinstance(sheet_name, int):
        ws = wb.worksheets[sheet_name]
    else:
        ws = wb[sheet_name]

    # Find last column and copy header style
    last_col = ws.max_column
    ref_cell = ws.cell(row=1, column=last_col)

    for col_offset, col_name in enumerate(
        ["three_point_qual_score", "five_point_qual_score"], start=1
    ):
        new_header = ws.cell(
            row=1,
            column=last_col + col_offset,
            value=col_name
        )
        new_header.font = copy(ref_cell.font)
        new_header.border = copy(ref_cell.border)
        new_header.fill = copy(ref_cell.fill)
        new_header.alignment = copy(ref_cell.alignment)

    # Write score values row by row
    for i in range(len(three_point_final)):
        ws.cell(row=i + 2, column=last_col + 1, value=three_point_final[i])
        ws.cell(row=i + 2, column=last_col + 2, value=five_point_final[i])

    wb.save(output_path)
    print(f"Done! Saved to: {output_path}")
    print(f"Sheets preserved: {wb.sheetnames}")

print("\nAll files processed successfully!")


Processing: /blue/prismap-ai-core/deehu/projects/QUAL_Score_Project/EPA_Narratives_Faculty.xlsx
Columns: ['ACGME ID', 'Assessment Event ID', 'Program', 'Activity', 'Activity Date', 'Date/Time of Assessment', 'Created By', 'Rater ACGME ID', 'Rater Role', 'Rater Job Title', 'Trainee Job Title', 'Trainee PGY (Current)', 'Trainee PGY (At Assessment)', 'Self Assessment', 'Phase', 'Complexity Value', 'Complexity Level', 'Entrustability Value', 'Entrustability Level', 'Feedback']
Total rows: 14700
Valid (non-junk) rows scored: 14661
Blank/junk rows skipped: 39
Done! Saved to: /blue/prismap-ai-core/deehu/projects/QUAL_Score_Project/EPA_Narratives_Faculty_QUAL_scored.xlsx
Sheets preserved: ['Sheet1']

Processing: /blue/prismap-ai-core/deehu/projects/QUAL_Score_Project/CASE_data.xlsx
Columns: ['ACGME ID', 'Assessment Event ID', 'Program', 'Activity', 'Activity Date', 'Date/Time of Assessment', 'Created By', 'Rater ACGME ID', 'Rater Role', 'Rater Job Title', 'Trainee Job Title', 'Trainee PGY (Cu